# Notebook 2: Robots Validation, Relevance-Guided Website Crawl, and Translation

This notebook replaces the previous one-layer flow:

`homepage → extract links → select top 1 → fetch → stop`

with a bounded graph traversal:

`homepage → discover internal links → score links → queue relevant unseen pages → fetch them → repeat`

The crawler uses:

- same-domain URL validation
- `robots.txt` checks for each crawled path
- a BFS queue ordered by depth
- cosine similarity to filter and rank links within each depth
- duplicate protection through `visited` and `queued` sets
- strict page, depth, and branching limits
- actual page-content scoring to select one final candidate per company


In [132]:
from __future__ import annotations

import asyncio
import csv
import heapq
import re
from collections import deque
from pathlib import Path
from typing import Any
from urllib.parse import urljoin, urlparse, urlunparse
from urllib.robotparser import RobotFileParser

import httpx
import numpy as np
import pandas as pd
import pycountry
import torch
from bs4 import BeautifulSoup
from langdetect import DetectorFactory, detect
from sentence_transformers import SentenceTransformer, util


## 1. Configuration


In [133]:
INPUT_COMPANY_CSV = Path("../data/processed/company_scrape_ready.csv")
ROBOTS_OUTPUT_CSV = Path("../data/processed/robot_check.csv")
HOMEPAGE_OUTPUT_CSV = Path("../data/processed/homepage_fetch.csv")
CRAWL_AUDIT_OUTPUT_CSV = Path("../data/processed/crawled_pages.csv")
PRODUCTION_FINAL_CSV = Path("../data/processed/production_final.csv")
TRANSLATION_OUTPUT_CSV = Path("../data/processed/production_500_sample.csv")
TRANSLATION_CACHE_CSV = Path("../data/processed/translation_cache.csv")

USER_AGENT = (
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/150.0.0.0 Safari/537.36"
)

REQUEST_TIMEOUT_SECONDS = 15
MAX_CONCURRENT_REQUESTS = 10
SECTION_HEADING_THEMES = [
    "our values",
    "core values",
    "company values",
    "our mission",
    "company mission",
    "our vision",
    "company vision",
    "our purpose",
    "company purpose",
    "our principles",
    "guiding principles",
    "our beliefs",
    "what we believe",
    "what we stand for",
    "our culture",
    "company culture",
    "our philosophy",
    "our ethos",
    "our identity",
    "our DNA",
    "how we work",
]

MIN_HEADING_SIMILARITY = 0.38
MAX_HEADING_CHARACTERS = 150

MAX_SECTION_ELEMENTS = 15
MAX_SECTION_CHARACTERS = 4_000
MIN_SECTION_CHARACTERS = 80
# Crawl limits prevent runaway websites, pagination traps, and excessive requests.
MAX_CRAWL_DEPTH = 3
MAX_PAGES_PER_COMPANY = 12
MAX_CHILDREN_PER_PAGE = 2

MIN_CONTENT_SIMILARITY = 0.42
MIN_SEMANTIC_MARGIN = 0.08
MIN_LINK_SIMILARITY = 0.38

MIN_PAGE_TEXT_CHARACTERS = 300
MIN_PAGE_TEXT_WORDS = 40

MAX_PAGE_TEXT_CHARACTERS = 10_000
CONTENT_SCORING_CHARACTERS = 10_000

# One variable controls how many companies are processed.
# Use a positive integer such as 5, 50, or 500.
# Use "full" to process every company.
RUN_SIZE: int | str = 'full'

# Set a flag to True only when that stage must be run again.
RERUN_ROBOTS = False
RERUN_HOMEPAGES = False
RERUN_CRAWL = False
RERUN_TRANSLATION = False

RANDOM_SEED = 42

MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

TARGET_LINK_THEMES = [
    "our core values",
    "company core values",
    "corporate values",
    "organisational values",
    "organizational values",
    "company principles",
    "guiding principles",
    "core principles",
    "our beliefs",
    "company beliefs",
    "what we believe",
    "our purpose",
    "company purpose",
    "our mission",
    "company mission",
    "our vision",
    "company vision",
    "mission vision and values",
    "purpose mission vision and values",
    "our culture",
    "company culture",
    "workplace culture",
    "organisational culture",
    "organizational culture",
    "shared values",
    "values and behaviours",
    "values and behaviors",
    "our commitments",
    "company commitments",
    "ethical principles",
    "business ethics",
    "code of values",
    "principles that guide us",
    "what guides our decisions",
    "how we work",
    "how we behave",
    "who we are and what we stand for",
    "what we stand for",
    "our identity and values",
    "our philosophy",
    "company philosophy",
    "our ethos",
    "corporate ethos",
    "our DNA",
    "company DNA",
    "our way",
    "the way we work",
    "our standards",
    "company standards",
    "our foundations",
    "our values and culture",
    "our mission and values",
    "our vision and values",
    "our purpose and values",
    "our principles and values",
]
NEGATIVE_CONTENT_THEMES = [
    (
        "A page advertising leadership training, management development, "
        "coaching, workshops, academies, courses, or professional education services."
    ),
    (
        "A page describing a commercial training programme, academy, consulting "
        "service, learning product, or customer offering."
    ),
    (
        "A page explaining how a service helps customers develop leadership, "
        "culture, vision, behaviours, purpose, or organisational values."
    ),
    (
        "The page mainly discusses corporate governance, board structure, "
        "executive oversight, committees, compliance, or shareholder responsibilities."
    ),
    (
        "The page mainly contains investor relations information, annual reports, "
        "financial performance, stock information, or corporate disclosures."
    ),
    (
        "The page mainly provides contact details, office locations, enquiry forms, "
        "phone numbers, email addresses, or customer support information."
    ),
    (
        "The page mainly lists jobs, vacancies, recruitment information, employee "
        "benefits, or career opportunities."
    ),
    (
        "The page mainly describes products, services, industries, capabilities, "
        "customer solutions, or commercial offerings."
    ),
    (
        "The page mainly contains news articles, press releases, media updates, "
        "events, announcements, or blog posts."
    ),
]

TARGET_CONTENT_THEMES = [
    (
        "The company explicitly explains its core values, guiding principles, "
        "beliefs, or standards that influence employee behaviour and business decisions."
    ),
    (
        "The page describes the organisation's mission, vision, and purpose, "
        "including what the company aims to achieve and why it exists."
    ),
    (
        "The company presents named values such as integrity, respect, innovation, "
        "collaboration, accountability, trust, excellence, inclusion, or sustainability."
    ),
    (
        "The page explains the behaviours expected from employees and how the "
        "organisation's values guide everyday work."
    ),
    (
        "The company describes its organisational culture, shared values, workplace "
        "beliefs, and the principles that shape how people work together."
    ),
    (
        "The page contains a structured list of company values, each supported by "
        "an explanation, definition, example, or behavioural statement."
    ),
    (
        "The company explains what it stands for, what it believes in, and the "
        "principles used to guide choices and relationships with stakeholders."
    ),
    (
        "The organisation defines its ethical principles, commitments, or standards "
        "of conduct as part of its identity and culture."
    ),
    (
        "The page explains the company's philosophy, ethos, identity, or DNA in "
        "terms of values, beliefs, purpose, and expected behaviour."
    ),
    (
        "The company describes how its mission, purpose, vision, culture, and values "
        "connect to its long-term direction and decision-making."
    ),
]

SECTION_VERIFICATION_THEMES = [
    (
        "The organisation explicitly states its own core values, beliefs, "
        "principles, mission, vision, purpose, or expected behaviours."
    ),
    (
        "This section describes what the company itself stands for and how "
        "its employees or leaders are expected to behave."
    ),
    (
        "This section presents the organisation's identity, culture, values, "
        "mission, vision, or guiding principles."
    ),
]

SECTION_REJECTION_THEMES = [
    (
        "This section describes company history, restructuring, reorganisation, "
        "business divisions, factories, milestones, anniversaries, or past events."
    ),
    (
        "This section describes governance structures, boards, committees, "
        "directors, reporting responsibilities, ESG governance, or oversight bodies."
    ),
    (
        "This section is specifically about a product specification, "
        "technical feature, equipment model, product catalogue, or service description."
    ),
    (
        "This section is specifically about a project, event, news story, "
        "programme, gallery, membership, or temporary initiative."
    ),
    (
        "This section contains legal conditions, contractual clauses, privacy "
        "terms, prices, monetary value, inventory value, or compliance declarations."
    ),
    (
        "This section mainly lists addresses, phone numbers, email details, "
        "office locations, staff members, executives, or organisational roles."
    ),
    (
        "This section mainly describes employment vacancies, job applications, "
        "career opportunities, or recruitment instructions."
    ),
]

EXCLUDED_PATH_PATTERNS = [
    r"/contact(?:/|$)",
    r"/privacy(?:/|$)",
    r"/terms(?:/|$)",
    r"/terms-of-use(?:/|$)",
    r"/termos-de-uso(?:/|$)",
    r"/legal(?:/|$)",
    r"/login(?:/|$)",
    r"/signin(?:/|$)",
    r"/search(?:/|$)",
    r"/cart(?:/|$)",
    r"/checkout(?:/|$)",
    r"/author(?:/|$)",
    r"/tag(?:/|$)",
    r"/category(?:/|$)",
    r"/news(?:/|$)",
    r"/blog(?:/|$)",
    r"/press(?:/|$)",
    r"/media(?:/|$)",
    r"/jobs?(?:/|$)",
    r"/careers?(?:/|$)",
    r"/vacanc(?:y|ies)(?:/|$)",
    r"/recruitment(?:/|$)",
    r"/board-of-directors(?:/|$)",
    r"/leadership-team(?:/|$)",
    r"/management-team(?:/|$)",
    r"/corporate-governance(?:/|$)",
    r"/investor-relations(?:/|$)",
    r"/awards?(?:/|$)",
    r"/milestones?(?:/|$)",
    r"/clients?(?:/|$)",
    r"/customers?(?:/|$)",
    r"/partners?(?:/|$)",
    r"/brands?(?:/|$)",
    r"/franchises?(?:/|$)",
    r"/groupcompanies(?:/|$)",
    r"/group-companies(?:/|$)",
    r"/team(?:/|$)",
    r"/our-team(?:/|$)",
    r"/people(?:/|$)",
    r"/certifications?(?:/|$)",
    r"/accreditations?(?:/|$)",
    r"/quality-assurance(?:/|$)",
    r"/quality(?:/|$)",
    r"/products?(?:/|$)",
    r"/services?(?:/|$)",
    r"/solutions?(?:/|$)",
    r"/industries?(?:/|$)",
    r"/case-studies?(?:/|$)",
    r"/resources?(?:/|$)",
    r"/events?(?:/|$)",
    r"/mental-health(?:/|$)",
    r"/governance(?:/|$)",
    r"/governanca(?:/|$)",
    r"/company-profile(?:/|$)",
    r"/executive-committee(?:/|$)",
    r"/quality-assurance(?:/|$)",
    r"/service(?:/|$)",
    r"/services?(?:/|$)",
    r"/noticias?(?:/|$)",
    r"/news(?:/|$)",
]

EXCLUDED_EXTENSIONS = {
    ".pdf", ".jpg", ".jpeg", ".png", ".gif", ".svg",
    ".zip", ".doc", ".docx", ".xls", ".xlsx", ".ppt", ".pptx",
}

ANCHOR_TEXT_NOISE = {
    "", "skip to content", "skip to main content", "click here",
    "read more", "learn more", "more", "menu", "close",
    "next", "previous", "back", "top",
}

for path in [
    ROBOTS_OUTPUT_CSV,
    HOMEPAGE_OUTPUT_CSV,
    CRAWL_AUDIT_OUTPUT_CSV,
    PRODUCTION_FINAL_CSV,
    TRANSLATION_OUTPUT_CSV,
    TRANSLATION_CACHE_CSV,
]:
    path.parent.mkdir(parents=True, exist_ok=True)


## 2. Load and prepare company input


In [134]:
df_companies_raw = pd.read_csv(INPUT_COMPANY_CSV)

required_columns = [
    "company_id",
    "name",
    "country",
    "size_bucket",
    "domain",
    "final_url",
]

missing_columns = [
    column for column in required_columns
    if column not in df_companies_raw.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

df_companies = (
    df_companies_raw[required_columns]
    .dropna(subset=["company_id", "final_url"])
    .drop_duplicates(subset=["company_id"], keep="first")
    .reset_index(drop=True)
)

if isinstance(RUN_SIZE, int):
    if RUN_SIZE <= 0:
        raise ValueError("RUN_SIZE must be greater than 0.")

    df_companies = (
        df_companies.sample(
            n=min(RUN_SIZE, len(df_companies)),
            random_state=RANDOM_SEED,
        )
        .sort_values("company_id")
        .reset_index(drop=True)
    )

elif isinstance(RUN_SIZE, str) and RUN_SIZE.lower() == "full":
    df_companies = df_companies.reset_index(drop=True)

else:
    raise ValueError("RUN_SIZE must be a positive integer or 'full'.")

print("Companies selected:", len(df_companies))
df_companies.head(3)


Companies selected: 1449


,company_id,name,country,size_bucket,domain,final_url
0,C0001,froneri,australia,enterprise,froneri.com,https://www.froneri.com/
1,C0002,gloria jean’s coffees australia,australia,enterprise,gloriajeanscoffees.com.au,https://www.gloriajeans.com.au/
2,C0003,australian subscription television & radio ass...,australia,enterprise,astra.org.au,https://astra.org.au


## 3. Shared URL, HTML, and feature helpers


In [135]:
def normalise_domain(url: str) -> str:
    parsed_url = urlparse(str(url))
    domain = parsed_url.netloc.lower().strip()

    if domain.startswith("www."):
        domain = domain[4:]

    return domain


def clean_internal_url(url: str) -> str:
    parsed_url = urlparse(str(url).strip())

    cleaned_url = parsed_url._replace(
        query="",
        fragment="",
    )

    return urlunparse(cleaned_url).rstrip("/")


def get_url_path(url: str) -> str:
    path = urlparse(str(url)).path
    return path if path else "/"


def is_same_homepage_domain(url: str, homepage_url: str) -> bool:
    link_domain = normalise_domain(url)
    homepage_domain = normalise_domain(homepage_url)

    return (
        link_domain == homepage_domain
        or link_domain.endswith("." + homepage_domain)
    )


def clean_anchor_text(anchor_text: str) -> str:
    cleaned = re.sub(r"\s+", " ", str(anchor_text)).strip()
    return cleaned[:300]


def clean_page_text(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")

    for tag in soup([
        "script",
        "style",
        "noscript",
        "header",
        "footer",
        "nav",
        "form",
        "aside",
        "iframe",
        "svg",
        "button",
    ]):
        tag.decompose()

    text = soup.get_text(separator=" ", strip=True)
    text = re.sub(r"\s+", " ", text)

    return text[:MAX_PAGE_TEXT_CHARACTERS]

def extract_candidate_sections_from_html(
    html: str,
    max_elements: int = MAX_SECTION_ELEMENTS,
    max_characters: int = MAX_SECTION_CHARACTERS,
) -> list[dict[str, Any]]:
    """
    Find semantically relevant headings in any supported language and
    extract the local HTML section following each heading.
    """
    soup = BeautifulSoup(html, "html.parser")

    for tag in soup([
        "script",
        "style",
        "noscript",
        "header",
        "footer",
        "nav",
        "form",
        "aside",
        "iframe",
        "svg",
        "button",
    ]):
        tag.decompose()

    heading_names = ["h1", "h2", "h3", "h4", "h5", "h6"]
    headings = soup.find_all(heading_names)

    heading_records: list[dict[str, Any]] = []
    heading_texts: list[str] = []

    for heading in headings:
        heading_text = re.sub(
            r"\s+",
            " ",
            heading.get_text(" ", strip=True),
        ).strip()

        if not heading_text:
            continue

        heading_records.append({
            "element": heading,
            "heading_text": heading_text,
        })
        heading_texts.append(heading_text)

    if not heading_records:
        return []

    heading_embeddings = model.encode(
        heading_texts,
        convert_to_tensor=True,
        normalize_embeddings=True,
    )

    heading_scores = util.cos_sim(
        heading_embeddings,
        section_heading_theme_embeddings,
    )

    max_scores, best_theme_indexes = torch.max(
        heading_scores,
        dim=1,
    )

    candidate_sections: list[dict[str, Any]] = []
    heading_tag_set = set(heading_names)

    for record, score, theme_index in zip(
        heading_records,
        max_scores.cpu().tolist(),
        best_theme_indexes.cpu().tolist(),
    ):
        heading_similarity = float(score)

        if heading_similarity < MIN_HEADING_SIMILARITY:
            continue

        heading = record["element"]
        heading_text = record["heading_text"]

        section_parts = [heading_text]
        current = heading.find_next_sibling()
        collected_elements = 0

        while current is not None and collected_elements < max_elements:
            if current.name in heading_tag_set:
                break

            text = re.sub(
                r"\s+",
                " ",
                current.get_text(" ", strip=True),
            ).strip()

            if text:
                section_parts.append(text)

            current = current.find_next_sibling()
            collected_elements += 1

        section_text = re.sub(
            r"\s+",
            " ",
            " ".join(section_parts),
        ).strip()

        section_text = section_text[:max_characters]

        if len(section_text) < MIN_SECTION_CHARACTERS:
            continue

        candidate_sections.append({
            "section_heading": heading_text,
            "section_text": section_text,
            "heading_similarity_score": heading_similarity,
            "heading_matched_theme": SECTION_HEADING_THEMES[
                theme_index
            ],
        })

    return candidate_sections


def is_excluded_url(url: str) -> bool:
    path = get_url_path(url).lower()

    if any(path.endswith(extension) for extension in EXCLUDED_EXTENSIONS):
        return True

    return any(
        re.search(pattern, path)
        for pattern in EXCLUDED_PATH_PATTERNS
    )


def improved_extract_path_features(url_path: str) -> str:
    if pd.isna(url_path):
        return ""

    path = str(url_path).lower().strip()
    path = re.sub(r"\.[a-zA-Z0-9]+$", "", path)

    noise_words = [
        "index",
        "home",
        "page",
        "wp-content",
        "uploads",
        "assets",
        "attachment",
    ]

    for noise_word in noise_words:
        path = path.replace(noise_word, " ")

    words = re.sub(r"[/_-]+", " ", path)

    abbreviations = {
    r"\bco\b": "company",
    r"\babt\b": "about",
    r"\bprofile\b": "company profile",
    r"\bvis\b": "vision",
    r"\bval\b": "values",
}

    for pattern, replacement in abbreviations.items():
        words = re.sub(pattern, replacement, words)

    return " ".join(words.split())


def build_feature_text(anchor_text: str, url_path: str) -> str:
    anchor = clean_anchor_text(anchor_text)
    path_features = improved_extract_path_features(url_path)

    if anchor.lower() in ANCHOR_TEXT_NOISE:
        anchor = ""

    if anchor and path_features:
        return f"{anchor} ({path_features})"

    return anchor or path_features


def extract_internal_links_from_page(
    html: str,
    current_page_url: str,
    allowed_domain_url: str,
) -> list[dict[str, str]]:
    soup = BeautifulSoup(html, "html.parser")
    links: list[dict[str, str]] = []

    for anchor in soup.find_all("a", href=True):
        raw_link = anchor.get("href")

        if not raw_link:
            continue

        absolute_url = urljoin(current_page_url, raw_link)
        internal_url = clean_internal_url(absolute_url)

        if not internal_url.startswith(("https://", "http://")):
            continue

        if not is_same_homepage_domain(internal_url, allowed_domain_url):
            continue

        if is_excluded_url(internal_url):
            continue

        links.append({
            "raw_link": raw_link,
            "internal_url": internal_url,
            "url_path": get_url_path(internal_url),
            "anchor_text": clean_anchor_text(anchor.get_text(" ")),
        })

    # Preserve the first useful anchor while removing duplicate destination URLs.
    unique_links: list[dict[str, str]] = []
    seen_urls: set[str] = set()

    for link in links:
        if link["internal_url"] in seen_urls:
            continue

        seen_urls.add(link["internal_url"])
        unique_links.append(link)

    return unique_links


## 4. Ticket 10268: robots.txt validation


In [ ]:
def build_robots_url(final_url: str) -> str:
    parsed_url = urlparse(final_url)
    return f"{parsed_url.scheme}://{parsed_url.netloc}/robots.txt"


async def fetch_robots_text(
    client: httpx.AsyncClient,
    robots_url: str,
) -> tuple[int | None, str | None, str | None]:
    try:
        response = await client.get(
            robots_url,
            timeout=REQUEST_TIMEOUT_SECONDS,
        )

        return response.status_code, response.text, None

    except httpx.HTTPError as error:
        return None, None, f"robots_fetch_error_{type(error).__name__}"


def build_robots_parser(
    robots_url: str,
    robots_text: str | None,
) -> RobotFileParser | None:
    if not robots_text:
        return None

    parser = RobotFileParser()
    parser.set_url(robots_url)
    parser.parse(robots_text.splitlines())

    return parser


async def validate_company_robots(
    company: pd.Series,
    client: httpx.AsyncClient,
    request_semaphore: asyncio.Semaphore,
) -> dict[str, Any]:
    final_url = company["final_url"]
    robots_url = build_robots_url(final_url)

    async with request_semaphore:
        status_code, robots_text, fetch_error = await fetch_robots_text(
            client=client,
            robots_url=robots_url,
        )

    if fetch_error:
        robots_allowed = True
        robots_reason = fetch_error
    elif status_code is not None and status_code >= 400:
        robots_allowed = True
        robots_reason = f"robots_status_{status_code}"
    else:
        try:
            parser = build_robots_parser(robots_url, robots_text)
            robots_allowed = (
                True
                if parser is None
                else parser.can_fetch(USER_AGENT, final_url)
            )
            robots_reason = "allowed" if robots_allowed else "blocked"
        except Exception as error:
            robots_allowed = True
            robots_reason = f"robots_parse_error_{type(error).__name__}"

    return {
        "company_id": company["company_id"],
        "name": company["name"],
        "country": company["country"],
        "size_bucket": company["size_bucket"],
        "domain": company["domain"],
        "final_url": final_url,
        "robots_url": robots_url,
        "robots_status_code": status_code,
        "robots_allowed": robots_allowed,
        "robots_reason": robots_reason,
    }


async def run_robots_validation(df_input: pd.DataFrame) -> pd.DataFrame:
    request_semaphore = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)

    async with httpx.AsyncClient(
        headers={"User-Agent": USER_AGENT},
        follow_redirects=True,
    ) as client:
        tasks = [
            validate_company_robots(
                company=row,
                client=client,
                request_semaphore=request_semaphore,
            )
            for _, row in df_input.iterrows()
        ]

        results = await asyncio.gather(*tasks)

    return pd.DataFrame(results)
if ROBOTS_OUTPUT_CSV.exists() and not RERUN_ROBOTS:
    df_robot_cached = pd.read_csv(ROBOTS_OUTPUT_CSV)

    requested_company_ids = set(df_companies["company_id"])
    cached_company_ids = set(df_robot_cached["company_id"])
    missing_company_ids = requested_company_ids - cached_company_ids

    df_robot_validated = df_robot_cached[
        df_robot_cached["company_id"].isin(requested_company_ids)
    ].copy()

    if missing_company_ids:
        df_missing_robots = df_companies[
            df_companies["company_id"].isin(missing_company_ids)
        ].copy()

        df_new_robots = await run_robots_validation(df_missing_robots)

        df_robot_validated = pd.concat(
            [df_robot_validated, df_new_robots],
            ignore_index=True,
        )

        df_robot_cached = pd.concat(
            [df_robot_cached, df_new_robots],
            ignore_index=True,
        ).drop_duplicates(
            subset=["company_id"],
            keep="last",
        )

        df_robot_cached.to_csv(
            ROBOTS_OUTPUT_CSV,
            index=False,
        )

    print("Loaded robots results from cache.")

else:
    df_robot_validated = await run_robots_validation(df_companies)
    df_robot_validated.to_csv(ROBOTS_OUTPUT_CSV, index=False)

print("Robots allowed:", int(df_robot_validated["robots_allowed"].sum()))
print("Robots blocked:", int((~df_robot_validated["robots_allowed"]).sum()))
df_robot_validated.head(3)


## 5. Ticket 10269: homepage validation


In [ ]:
async def fetch_company_homepage(
    company: pd.Series,
    client: httpx.AsyncClient,
    request_semaphore: asyncio.Semaphore,
) -> dict[str, Any]:
    final_url = company["final_url"]

    try:
        async with request_semaphore:
            response = await client.get(
                final_url,
                timeout=REQUEST_TIMEOUT_SECONDS,
            )

        status_code = response.status_code
        content_type = response.headers.get("Content-Type", "").lower()

        if status_code >= 400:
            error = f"homepage_http_error_{status_code}"
        elif "text/html" not in content_type:
            error = f"homepage_not_html_{content_type}"
        else:
            error = ""

        return {
            "company_id": company["company_id"],
            "homepage_status_code": status_code,
            "homepage_fetch_url": clean_internal_url(str(response.url)),
            "homepage_error": error,
        }

    except httpx.HTTPError as error:
        return {
            "company_id": company["company_id"],
            "homepage_status_code": None,
            "homepage_fetch_url": None,
            "homepage_error": f"homepage_fetch_error_{type(error).__name__}",
        }


async def run_homepage_fetch(df_input: pd.DataFrame) -> pd.DataFrame:
    request_semaphore = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)

    async with httpx.AsyncClient(
        headers={"User-Agent": USER_AGENT},
        follow_redirects=True,
    ) as client:
        tasks = [
            fetch_company_homepage(
                company=row,
                client=client,
                request_semaphore=request_semaphore,
            )
            for _, row in df_input.iterrows()
        ]

        results = await asyncio.gather(*tasks)

    return pd.DataFrame(results)
df_homepage_input = df_robot_validated.loc[
    df_robot_validated["robots_allowed"]
].copy()

if HOMEPAGE_OUTPUT_CSV.exists() and not RERUN_HOMEPAGES:
    df_homepage_cached = pd.read_csv(HOMEPAGE_OUTPUT_CSV)

    requested_company_ids = set(df_homepage_input["company_id"])
    cached_company_ids = set(df_homepage_cached["company_id"])
    missing_company_ids = requested_company_ids - cached_company_ids

    df_homepage_fetch = df_homepage_cached[
        df_homepage_cached["company_id"].isin(requested_company_ids)
    ].copy()

    if missing_company_ids:
        df_missing_homepages = df_homepage_input[
            df_homepage_input["company_id"].isin(missing_company_ids)
        ].copy()

        df_new_homepage_results = await run_homepage_fetch(
            df_missing_homepages
        )

        df_new_homepage_fetch = df_missing_homepages.merge(
            df_new_homepage_results,
            on="company_id",
            how="left",
            validate="one_to_one",
        )

        df_homepage_fetch = pd.concat(
            [df_homepage_fetch, df_new_homepage_fetch],
            ignore_index=True,
        )

        df_homepage_cached = pd.concat(
            [df_homepage_cached, df_new_homepage_fetch],
            ignore_index=True,
        ).drop_duplicates(
            subset=["company_id"],
            keep="last",
        )

        df_homepage_cached.to_csv(
            HOMEPAGE_OUTPUT_CSV,
            index=False,
        )

    print("Loaded homepage results from cache.")

else:
    df_homepage_results = await run_homepage_fetch(df_homepage_input)

    df_homepage_fetch = df_homepage_input.merge(
        df_homepage_results,
        on="company_id",
        how="left",
        validate="one_to_one",
    )

    df_homepage_fetch.to_csv(
        HOMEPAGE_OUTPUT_CSV,
        index=False,
    )

print(
    "Valid homepages:",
    int(df_homepage_fetch["homepage_error"].fillna("").eq("").sum()),
)
df_homepage_fetch.head(3)


Loaded homepage results from cache.
Valid homepages: 19


,company_id,name,country,size_bucket,domain,final_url,robots_url,robots_status_code,robots_allowed,robots_reason,homepage_status_code,homepage_fetch_url,homepage_error
10,C0050,southern steel group,australia,large,southernsteelgroup.com.au,https://www.southernsteelgroup.com.au/,https://www.southernsteelgroup.com.au/robots.txt,200.0,True,allowed,200.0,https://www.southernsteelgroup.com.au,NaN
41,C0142,the rabbit sanctuary,australia,small,rabbitsanctuary.com.au,https://www.rabbitsanctuary.com.au,https://www.rabbitsanctuary.com.au/robots.txt,200.0,True,allowed,200.0,https://www.rabbitsanctuary.com.au,NaN
47,C0169,grupo ccr,brazil,enterprise,grupoccr.com.br,https://www.motiva.com.br/,https://www.motiva.com.br/robots.txt,200.0,True,allowed,200.0,https://www.motiva.com.br,NaN


## 6. Load the embedding model once


In [ ]:
model = SentenceTransformer(MODEL_NAME)

link_theme_embeddings = model.encode(
    TARGET_LINK_THEMES,
    convert_to_tensor=True,
    normalize_embeddings=True,
)

positive_content_theme_embeddings = model.encode(
    TARGET_CONTENT_THEMES,
    convert_to_tensor=True,
    normalize_embeddings=True,
)

negative_content_theme_embeddings = model.encode(
    NEGATIVE_CONTENT_THEMES,
    convert_to_tensor=True,
    normalize_embeddings=True,
)
section_heading_theme_embeddings = model.encode(
    SECTION_HEADING_THEMES,
    convert_to_tensor=True,
    normalize_embeddings=True,
)
section_verification_embeddings = model.encode(
    SECTION_VERIFICATION_THEMES,
    convert_to_tensor=True,
    normalize_embeddings=True,
)

section_rejection_embeddings = model.encode(
    SECTION_REJECTION_THEMES,
    convert_to_tensor=True,
    normalize_embeddings=True,
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 19394.16it/s]


## 7. Reusable semantic scoring


In [ ]:
def score_discovered_links(
    links: list[dict[str, str]],
) -> list[dict[str, Any]]:
    if not links:
        return []

    valid_links: list[dict[str, str]] = []
    feature_texts: list[str] = []

    for link in links:
        feature_text = build_feature_text(
            anchor_text=link["anchor_text"],
            url_path=link["url_path"],
        )

        if not feature_text.strip():
            continue

        valid_links.append(link)
        feature_texts.append(feature_text)

    if not valid_links:
        return []

    feature_embeddings = model.encode(
        feature_texts,
        convert_to_tensor=True,
        normalize_embeddings=True,
    )

    cosine_scores = util.cos_sim(
        feature_embeddings,
        link_theme_embeddings,
    )

    max_scores, best_theme_indexes = torch.max(
        cosine_scores,
        dim=1,
    )

    scored_links: list[dict[str, Any]] = []

    for link, feature_text, score, theme_index in zip(
        valid_links,
        feature_texts,
        max_scores.cpu().tolist(),
        best_theme_indexes.cpu().tolist(),
    ):
        scored_links.append({
            **link,
            "feature_text": feature_text,
            "similarity_score": float(score),
            "matched_theme": TARGET_LINK_THEMES[theme_index],
        })

    return scored_links


def split_text_into_chunks(
    text: str,
    chunk_size: int = 1_500,
    overlap: int = 250,
) -> list[str]:
    cleaned_text = re.sub(r"\s+", " ", str(text)).strip()

    if not cleaned_text:
        return []

    chunks: list[str] = []
    start = 0

    while start < len(cleaned_text):
        end = min(start + chunk_size, len(cleaned_text))
        chunk = cleaned_text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end >= len(cleaned_text):
            break

        start = end - overlap

    return chunks


def score_page_content(
    page_text: str,
) -> dict[str, float]:
    chunks = split_text_into_chunks(
        str(page_text)[:CONTENT_SCORING_CHARACTERS]
    )

    if not chunks:
        return {
            "content_positive_score": float("nan"),
            "content_negative_score": float("nan"),
            "semantic_margin": float("nan"),
        }

    chunk_embeddings = model.encode(
        chunks,
        convert_to_tensor=True,
        normalize_embeddings=True,
    )

    positive_scores = util.cos_sim(
        chunk_embeddings,
        positive_content_theme_embeddings,
    ).max(dim=1).values

    negative_scores = util.cos_sim(
        chunk_embeddings,
        negative_content_theme_embeddings,
    ).max(dim=1).values

    chunk_margins = positive_scores - negative_scores

    best_chunk_index = int(
        torch.argmax(chunk_margins).item()
    )

    return {
        "content_positive_score": float(
            positive_scores[best_chunk_index].cpu().item()
        ),
        "content_negative_score": float(
            negative_scores[best_chunk_index].cpu().item()
        ),
        "semantic_margin": float(
            chunk_margins[best_chunk_index].cpu().item()
        ),
    }

def select_best_candidate_section(
    html: str,
) -> dict[str, Any]:
    sections = extract_candidate_sections_from_html(html)

    if not sections:
        return {
            "candidate_section_heading": "",
            "candidate_section_text": "",
            "heading_similarity_score": np.nan,
            "heading_matched_theme": "",
            "section_positive_score": np.nan,
            "section_negative_score": np.nan,
            "section_semantic_margin": np.nan,
            "candidate_section_count": 0,
        }

    feature_texts = [
        (
            f"Heading: {section['section_heading']}. "
            f"Section: {section['section_text'][:1_500]}"
        )
        for section in sections
    ]

    embeddings = model.encode(
        feature_texts,
        convert_to_tensor=True,
        normalize_embeddings=True,
    )

    positive_scores = util.cos_sim(
        embeddings,
        section_verification_embeddings,
    ).max(dim=1).values

    negative_scores = util.cos_sim(
        embeddings,
        section_rejection_embeddings,
    ).max(dim=1).values
    margins = positive_scores - negative_scores

    heading_scores = torch.tensor(
        [
            section["heading_similarity_score"]
            for section in sections
        ],
        dtype=positive_scores.dtype,
        device=positive_scores.device,
    )

    selection_scores = (
        margins
        + (0.20 * heading_scores)
        + (0.10 * positive_scores)
    )

    best_index = int(
        torch.argmax(selection_scores).item()
    )

   
    best_section = sections[best_index]

    return {
        "candidate_section_heading": (
            best_section["section_heading"]
        ),
        "candidate_section_text": (
            best_section["section_text"]
        ),
         "heading_similarity_score": float(
        best_section["heading_similarity_score"]
        ),
        "heading_matched_theme": (
            best_section["heading_matched_theme"]
        ),
        "section_positive_score": float(
            positive_scores[best_index].cpu().item()
        ),
        "section_negative_score": float(
            negative_scores[best_index].cpu().item()
        ),
        "section_semantic_margin": float(
            margins[best_index].cpu().item()
        ),
        "candidate_section_count": len(sections),
    }
VALUE_SECTION_PATTERNS = [
    r"\bour values\b",
    r"\bcore values\b",
    r"\bcompany values\b",
    r"\bour principles\b",
    r"\bguiding principles\b",
    r"\bwhat we believe\b",
    r"\bwhat we stand for\b",
    r"\bmission.{0,40}vision.{0,40}values\b",
    r"\bpurpose.{0,40}values\b",
    r"\bvalues and behaviours\b",
    r"\bvalues and behaviors\b",
]

VALUE_WORD_PATTERNS = [
    r"\bintegrity\b",
    r"\brespect\b",
    r"\baccountability\b",
    r"\bexcellence\b",
    r"\bcollaboration\b",
    r"\binnovation\b",
    r"\btrust\b",
    r"\binclusion\b",
    r"\bsustainability\b",
    r"\bcommitment\b",
    r"\bcourage\b",
    r"\bsafety\b",
    r"\bcitizenship\b",
]


def calculate_value_evidence(text: str) -> dict[str, Any]:
    cleaned_text = re.sub(r"\s+", " ", str(text)).strip().lower()

    section_matches = sum(
        bool(re.search(pattern, cleaned_text))
        for pattern in VALUE_SECTION_PATTERNS
    )

    value_word_matches = sum(
        bool(re.search(pattern, cleaned_text))
        for pattern in VALUE_WORD_PATTERNS
    )

    has_explicit_value_section = section_matches >= 1
    has_multiple_named_values = value_word_matches >= 3

    accepted = (
        has_explicit_value_section
        or has_multiple_named_values
    )

    return {
        "value_section_matches": section_matches,
        "named_value_matches": value_word_matches,
        "contains_value_signal": accepted,
    }


## 8. Relevance-guided BFS crawler


In [ ]:
async def load_company_robots_parser(
    company: pd.Series,
    client: httpx.AsyncClient,
    request_semaphore: asyncio.Semaphore,
) -> RobotFileParser | None:
    robots_url = company["robots_url"]

    async with request_semaphore:
        status_code, robots_text, fetch_error = await fetch_robots_text(
            client=client,
            robots_url=robots_url,
        )

    if fetch_error or status_code is None or status_code >= 400:
        return None

    try:
        return build_robots_parser(robots_url, robots_text)
    except Exception:
        return None


async def crawl_company_website(
    company: pd.Series,
    client: httpx.AsyncClient,
    request_semaphore: asyncio.Semaphore,
) -> list[dict[str, Any]]:
    homepage_url = clean_internal_url(company["homepage_fetch_url"])

    robots_parser = await load_company_robots_parser(
        company=company,
        client=client,
        request_semaphore=request_semaphore,
    )

    queue = deque([
        {
            "url": homepage_url,
            "parent_url": None,
            "depth": 0,
            "discovery_score": 1.0,
            "matched_theme": "homepage",
            "anchor_text": "",
            "feature_text": "homepage",
        }
    ])

    visited: set[str] = set()
    queued: set[str] = {homepage_url}
    results: list[dict[str, Any]] = []

    while queue and len(visited) < MAX_PAGES_PER_COMPANY:
        current = queue.popleft()
        current_url = current["url"]
        queued.discard(current_url)

        if current_url in visited:
            continue

        if current["depth"] > MAX_CRAWL_DEPTH:
            continue

        if is_excluded_url(current_url):
            continue

        if (
            robots_parser is not None
            and not robots_parser.can_fetch(USER_AGENT, current_url)
        ):
            results.append({
                "company_id": company["company_id"],
                "name": company["name"],
                "country": company["country"],
                "domain": company["domain"],
                "url": current_url,
                "fetch_url": None,
                "size_bucket": company["size_bucket"],
                "parent_url": current["parent_url"],
                "depth": current["depth"],
                "anchor_text": current["anchor_text"],
                "feature_text": current["feature_text"],
                "discovery_score": current["discovery_score"],
                "matched_theme": current["matched_theme"],
                "status_code": None,
                "content_type": None,
                "page_text": "",
                "content_positive_score":np.nan,
                "content_negative_score":np.nan,
                "semantic_margin":np.nan,
                "links_discovered": 0,
                "links_queued": 0,
                "crawl_error": "robots_blocked",
                "candidate_section_heading": "",
                "candidate_section_text": "",
                "section_positive_score": np.nan,
                "section_negative_score": np.nan,
                "section_semantic_margin": np.nan,
                "candidate_section_count": 0,
                "heading_similarity_score": np.nan,
"heading_matched_theme": "",
            })
            visited.add(current_url)
            continue

        visited.add(current_url)

        try:
            async with request_semaphore:
                response = await client.get(
                    current_url,
                    timeout=REQUEST_TIMEOUT_SECONDS,
                )

            status_code = response.status_code
            content_type = response.headers.get("Content-Type", "").lower()
            fetch_url = clean_internal_url(str(response.url))
            if not is_same_homepage_domain(fetch_url, homepage_url):
                results.append({
                    "company_id": company["company_id"],
                    "name": company["name"],
                    "country": company["country"],
                    "domain": company["domain"],
                    "url": current_url,
                    "fetch_url": fetch_url,
                    "size_bucket": company["size_bucket"],
                    "parent_url": current["parent_url"],
                    "depth": current["depth"],
                    "anchor_text": current["anchor_text"],
                    "feature_text": current["feature_text"],
                    "discovery_score": current["discovery_score"],
                    "matched_theme": current["matched_theme"],
                    "status_code": response.status_code,
                    "content_type": response.headers.get("Content-Type", "").lower(),
                    "page_text": "",
                    "content_positive_score": np.nan,
                    "content_negative_score": np.nan,
                    "semantic_margin": np.nan,
                    "links_discovered": 0,
                    "links_queued": 0,
                    "crawl_error": "redirected_outside_company_domain",
                    "candidate_section_heading": "",
                    "candidate_section_text": "",
                    "section_positive_score": np.nan,
                    "section_negative_score": np.nan,
                    "section_semantic_margin": np.nan,
                    "candidate_section_count": 0,
                    "heading_similarity_score": np.nan,
"heading_matched_theme": "",
                })
                continue

            if status_code >= 400:
                crawl_error = f"http_error_{status_code}"
                page_text = ""
            elif "text/html" not in content_type:
                crawl_error = f"not_html_{content_type}"
                page_text = ""
            else:
                page_text = clean_page_text(response.text)

                page_character_count = len(page_text)
                page_word_count = len(page_text.split())

                if page_character_count < MIN_PAGE_TEXT_CHARACTERS:
                    crawl_error = "page_text_too_short"
                elif page_word_count < MIN_PAGE_TEXT_WORDS:
                    crawl_error = "page_word_count_too_low"
                else:
                    crawl_error = ""

            if crawl_error:
                results.append({
                    "company_id": company["company_id"],
                    "name": company["name"],
                    "country": company["country"],
                    "domain": company["domain"],
                    "url": current_url,
                    "fetch_url": fetch_url,
                    "size_bucket": company["size_bucket"],
                    "parent_url": current["parent_url"],
                    "depth": current["depth"],
                    "anchor_text": current["anchor_text"],
                    "feature_text": current["feature_text"],
                    "discovery_score": current["discovery_score"],
                    "matched_theme": current["matched_theme"],
                    "status_code": status_code,
                    "content_type": content_type,
                    "page_text": "",
                    "content_positive_score": np.nan,
                    "content_negative_score":np.nan,
                    "semantic_margin":np.nan,
                    "links_discovered": 0,
                    "links_queued": 0,
                    "crawl_error": crawl_error,
                    "candidate_section_heading": "",
                    "candidate_section_text": "",
                    "section_positive_score": np.nan,
                    "section_negative_score": np.nan,
                    "section_semantic_margin": np.nan,
                    "candidate_section_count": 0,
                    "heading_similarity_score": np.nan,
"heading_matched_theme": "",
                })
                continue

            content_score = score_page_content(page_text)

            section_result = select_best_candidate_section(
                response.text
            )

            child_links: list[dict[str, str]] = []
            selected_children: list[dict[str, Any]] = []

            if current["depth"] < MAX_CRAWL_DEPTH:
                child_links = extract_internal_links_from_page(
                    html=response.text,
                    current_page_url=fetch_url,
                    allowed_domain_url=homepage_url,
                )

                scored_children = score_discovered_links(child_links)

                eligible_children = [
                    child
                    for child in scored_children
                    if child["similarity_score"] >= MIN_LINK_SIMILARITY
                    and child["internal_url"] not in visited
                    and child["internal_url"] not in queued
                ]

                # Avoid sorting every child when only a small fixed number is needed.
                selected_children = heapq.nlargest(
                    MAX_CHILDREN_PER_PAGE,
                    eligible_children,
                    key=lambda child: child["similarity_score"],
                )

                for child in selected_children:
                    child_url = child["internal_url"]

                    queue.append({
                        "url": child_url,
                        "parent_url": fetch_url,
                        "depth": current["depth"] + 1,
                        "discovery_score": child["similarity_score"],
                        "matched_theme": child["matched_theme"],
                        "anchor_text": child["anchor_text"],
                        "feature_text": child["feature_text"],
                    })
                    queued.add(child_url)

            results.append({
                "company_id": company["company_id"],
                "name": company["name"],
                "country": company["country"],
                "domain": company["domain"],
                "url": current_url,
                "fetch_url": fetch_url,
                "size_bucket": company["size_bucket"],
                "parent_url": current["parent_url"],
                "depth": current["depth"],
                "anchor_text": current["anchor_text"],
                "feature_text": current["feature_text"],
                "discovery_score": current["discovery_score"],
                "matched_theme": current["matched_theme"],
                "status_code": status_code,
                "content_type": content_type,
                "page_text": page_text,
                "content_positive_score": (
                content_score["content_positive_score"]
                ),
                "content_negative_score": (
                    content_score["content_negative_score"]
                ),
                "semantic_margin": (
                    content_score["semantic_margin"]
                ),
               
                "links_discovered": len(child_links),
                "links_queued": len(selected_children),
                "crawl_error": "",
                "candidate_section_heading": (
                section_result["candidate_section_heading"]
                ),
                "candidate_section_text": (
                    section_result["candidate_section_text"]
                ),
                "section_positive_score": (
                    section_result["section_positive_score"]
                ),
                "section_negative_score": (
                    section_result["section_negative_score"]
                ),
                "section_semantic_margin": (
                    section_result["section_semantic_margin"]
                ),
                "candidate_section_count": (
                    section_result["candidate_section_count"]
                ),
                "heading_similarity_score": (
                section_result["heading_similarity_score"]
                ),
                "heading_matched_theme": (
                section_result["heading_matched_theme"]
                ),
            })

        except httpx.HTTPError as error:
            results.append({
                "company_id": company["company_id"],
                "name": company["name"],
                "country": company["country"],
                "domain": company["domain"],
                "url": current_url,
                "fetch_url": None,
                "size_bucket": company["size_bucket"],
                "parent_url": current["parent_url"],
                "depth": current["depth"],
                "anchor_text": current["anchor_text"],
                "feature_text": current["feature_text"],
                "discovery_score": current["discovery_score"],
                "matched_theme": current["matched_theme"],
                "status_code": None,
                "content_type": None,
                "page_text": "",
                "content_positive_score": np.nan,
                "content_negative_score": np.nan,
                "semantic_margin": np.nan,
                "links_discovered": 0,
                "links_queued": 0,
                "crawl_error": f"fetch_error_{type(error).__name__}",
                "candidate_section_heading": "",
                "candidate_section_text": "",
                "section_positive_score": np.nan,
                "section_negative_score": np.nan,
                "section_semantic_margin": np.nan,
                "candidate_section_count": 0,
                "heading_similarity_score": np.nan,
"heading_matched_theme": "",
            })

    return results


async def run_company_crawls(df_input: pd.DataFrame) -> pd.DataFrame:
    request_semaphore = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)

    async with httpx.AsyncClient(
        headers={"User-Agent": USER_AGENT},
        follow_redirects=True,
    ) as client:
        tasks = [
            crawl_company_website(
                company=row,
                client=client,
                request_semaphore=request_semaphore,
            )
            for _, row in df_input.iterrows()
        ]

        company_results = await asyncio.gather(*tasks)

    flattened_results = [
        page
        for company_pages in company_results
        for page in company_pages
    ]

    return pd.DataFrame(flattened_results)


## 9. Select crawl input and run


In [ ]:
df_crawl_input = df_homepage_fetch.loc[
    df_homepage_fetch["robots_allowed"]
    & df_homepage_fetch["homepage_error"].fillna("").eq("")
    & df_homepage_fetch["homepage_fetch_url"].notna()
].copy()

print("Companies selected for crawl:", len(df_crawl_input))

if CRAWL_AUDIT_OUTPUT_CSV.exists() and not RERUN_CRAWL:
    df_crawl_cached = pd.read_csv(CRAWL_AUDIT_OUTPUT_CSV)

    requested_company_ids = set(df_crawl_input["company_id"])
    cached_company_ids = set(df_crawl_cached["company_id"])
    missing_company_ids = requested_company_ids - cached_company_ids

    df_crawled_pages = df_crawl_cached[
        df_crawl_cached["company_id"].isin(requested_company_ids)
    ].copy()

    if missing_company_ids:
        df_missing_crawl = df_crawl_input[
            df_crawl_input["company_id"].isin(missing_company_ids)
        ].copy()

        df_new_crawled_pages = await run_company_crawls(
            df_missing_crawl
        )

        df_crawled_pages = pd.concat(
            [df_crawled_pages, df_new_crawled_pages],
            ignore_index=True,
        )

        df_crawl_cached = pd.concat(
            [df_crawl_cached, df_new_crawled_pages],
            ignore_index=True,
        ).drop_duplicates(
            subset=["company_id", "url"],
            keep="last",
        )

        df_crawl_cached.to_csv(
            CRAWL_AUDIT_OUTPUT_CSV,
            index=False,
        )

    print("Loaded crawl results from cache.")

else:
    df_crawled_pages = await run_company_crawls(df_crawl_input)
    df_crawled_pages.to_csv(CRAWL_AUDIT_OUTPUT_CSV, index=False)

print("Pages visited:", len(df_crawled_pages))
print("Companies crawled:", df_crawled_pages["company_id"].nunique())

df_crawled_pages[
    [
        "company_id",
        "depth",
        "url",
        "parent_url",
        "discovery_score",
        "content_positive_score",
        "content_negative_score",
        "semantic_margin",
        "links_discovered",
        "links_queued",
        "crawl_error",
    ]
].sort_values(
    ["company_id", "depth", "discovery_score"],
    ascending=[True, True, False],
).head(30)
successful_page_mask = (
    df_crawled_pages["crawl_error"].fillna("").eq("")
    & df_crawled_pages["page_text"].fillna("").str.strip().ne("")
)

rescored_pages = (
    df_crawled_pages.loc[
        successful_page_mask,
        "page_text",
    ]
    .apply(score_page_content)
    .apply(pd.Series)
)

df_crawled_pages.loc[
    successful_page_mask,
    [
        "content_positive_score",
        "content_negative_score",
        "semantic_margin",
    ],
] = rescored_pages.to_numpy()

df_crawled_pages.to_csv(
    CRAWL_AUDIT_OUTPUT_CSV,
    index=False,
)

print("Cached pages rescored with chunk scoring.")


Companies selected for crawl: 19
Pages visited: 156
Companies crawled: 19
Cached pages rescored with chunk scoring.


## 10. Select one final values-related page per company


In [ ]:
df_crawled_pages["excluded_final_url"] = (
    df_crawled_pages["url"]
    .fillna("")
    .apply(is_excluded_url)
)

df_crawled_pages = df_crawled_pages.loc[
    ~df_crawled_pages["excluded_final_url"]
].copy()

df_valid_pages = df_crawled_pages.loc[
    df_crawled_pages["crawl_error"].fillna("").eq("")
    & df_crawled_pages["page_text"].fillna("").str.strip().ne("")
    & df_crawled_pages[
        "candidate_section_text"
    ].fillna("").str.strip().ne("")
    & df_crawled_pages["section_positive_score"].notna()
    & df_crawled_pages["section_negative_score"].notna()
    & df_crawled_pages["section_semantic_margin"].notna()
    & (
        df_crawled_pages["section_positive_score"]
        >= MIN_CONTENT_SIMILARITY
    )
    & (
        df_crawled_pages["section_semantic_margin"]
        >= MIN_SEMANTIC_MARGIN
    )
].copy()

POSITIVE_VALUE_PATH_PATTERNS = [
    r"/values?(?:/|$)",
    r"/core-values?(?:/|$)",
    r"/our-values?(?:/|$)",
    r"/werte(?:/|$)",
    r"/nos-valeurs(?:/|$)",
    r"/valeurs(?:/|$)",
    r"/principles?(?:/|$)",
    r"/beliefs?(?:/|$)",
    r"/mission-and-values?(?:/|$)",
    r"/vision-and-values?(?:/|$)",
    r"/purpose-and-values?(?:/|$)",
    r"/mission-vision-values?(?:/|$)",
]


def calculate_value_path_bonus(url: str) -> float:
    path = get_url_path(url).lower()

    if any(
        re.search(pattern, path)
        for pattern in POSITIVE_VALUE_PATH_PATTERNS
    ):
        return 0.15

    return 0.0


df_valid_pages["value_path_bonus"] = (
    df_valid_pages["url"]
    .fillna("")
    .apply(calculate_value_path_bonus)
)

# Calculate text-based evidence for every candidate page
page_value_evidence = (
    df_valid_pages["candidate_section_text"]
    .apply(calculate_value_evidence)
    .apply(pd.Series)
)

df_valid_pages = pd.concat(
    [
        df_valid_pages.reset_index(drop=True),
        page_value_evidence.reset_index(drop=True),
    ],
    axis=1,
)

# Multilingual semantic acceptance rule
df_valid_pages["contains_value_signal"] = (
    # Strong explicit heading
    (
        (df_valid_pages["heading_similarity_score"] >= 0.70)
        & (df_valid_pages["section_positive_score"] >= 0.40)
        & (df_valid_pages["section_semantic_margin"] >= 0.05)
    )
    |
    # Moderate heading requires stronger contextual evidence
    (
        (df_valid_pages["heading_similarity_score"] >= 0.50)
        & (df_valid_pages["section_positive_score"] >= 0.42)
        & (df_valid_pages["section_semantic_margin"] >= 0.12)
    )
    |
    # Weak multilingual heading must have extremely strong section evidence
    (
        (df_valid_pages["heading_similarity_score"] >= 0.38)
        & (df_valid_pages["section_positive_score"] >= 0.50)
        & (df_valid_pages["section_semantic_margin"] >= 0.25)
    )
)

# Reject pages that do not have enough values-related evidence
df_valid_pages = df_valid_pages.loc[
    df_valid_pages["contains_value_signal"]
].copy()

# Rank the remaining valid pages
df_valid_pages["final_selection_score"] = (
    df_valid_pages["section_semantic_margin"]
    + df_valid_pages["value_path_bonus"]
    + (0.20 * df_valid_pages["heading_similarity_score"])
    + (0.10 * df_valid_pages["section_positive_score"])
    - (0.10 * df_valid_pages["section_negative_score"])
)

production_final_df = (
    df_valid_pages
    .sort_values(
        by=[
            "company_id",
            "final_selection_score",
            "section_semantic_margin",
            "section_positive_score",
            "depth",
        ],
        ascending=[
            True,
            False,
            False,
            False,
            True,
        ],
    )
    .drop_duplicates(
        subset=["company_id"],
        keep="first",
    )
    .rename(
        columns={
            "url": "internal_url",
            "fetch_url": "candidate_fetch_url",
            "status_code": "candidate_status_code",
            "page_text": "candidate_text_full",
            "candidate_section_text": "candidate_text",
            "crawl_error": "candidate_error",
        }
    )
    .reset_index(drop=True)
)

print("Final candidate pages:", len(production_final_df))
production_final_df[
    [
        "company_id",
        "name",
        "country",
        "depth",
        "internal_url",
        "discovery_score",
       "content_positive_score",
"content_negative_score",
"semantic_margin",
        "matched_theme",
    ]
].head(20)


Final candidate pages: 2


,company_id,name,country,depth,internal_url,discovery_score,content_positive_score,content_negative_score,semantic_margin,matched_theme
0,C0169,grupo ccr,brazil,3,https://www.motiva.com.br/esg/social,0.481552,0.481084,0.344246,0.136838,our ethos
1,C0989,daa international,saudi arabia,1,https://www.daa-international.com/about/what-w...,0.784450,0.458638,0.314525,0.144113,our vision


## 11. Extract a focused values-text window


In [ ]:
production_final_df["quality_status"] = "accepted"

production_final_df.to_csv(
    PRODUCTION_FINAL_CSV,
    index=False,
)


## 12. Translation configuration


In [ ]:
OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_MODEL = "translategemma:4b"
MAX_CONCURRENT_TRANSLATIONS = 2
MAX_RETRIES = 3
TRANSLATION_TIMEOUT_SECONDS = 120
DetectorFactory.seed = 0


## 13. Language detection and translation


In [ ]:
def detect_source_language(text: str) -> tuple[str, str]:
    detected_code = detect(text)

    code_aliases = {
        "zh-cn": "zh-Hans",
        "zh-tw": "zh-Hant",
    }

    detected_code = code_aliases.get(detected_code, detected_code)

    language_name_overrides = {
        "zh-Hans": "Chinese (Simplified)",
        "zh-Hant": "Chinese (Traditional)",
    }

    if detected_code in language_name_overrides:
        language_name = language_name_overrides[detected_code]
    else:
        base_code = detected_code.split("-")[0]
        language = pycountry.languages.get(alpha_2=base_code)
        language_name = language.name if language else detected_code

    return detected_code, language_name


def build_translation_prompt(
    text: str,
    source_code: str,
    source_language: str,
) -> str:
    return (
        f"Translate the following company values text from "
        f"{source_language} ({source_code}) into English.\n\n"
        "Rules:\n"
        "- Translate literally and completely.\n"
        "- Do not summarise.\n"
        "- Do not reorganise the content.\n"
        "- Do not add headings unless they exist in the source.\n"
        "- Do not explain the translation.\n"
        "- Return only the translated text.\n\n"
        f"SOURCE TEXT:\n{text}"
    )

async def translate_value(
    text: str,
    client: httpx.AsyncClient,
    semaphore: asyncio.Semaphore,
) -> dict[str, Any]:
    source_text = str(text).strip()

    try:
        source_code, source_language = detect_source_language(source_text)
    except Exception as error:
        return {
            "source_text": source_text,
            "detected_language": "unknown",
            "translation_needed": None,
            "translated_value_text": None,
            "translation_method": "not_started",
            "translation_status": "language_detection_failed",
            "translation_error": type(error).__name__,
        }

    if source_code == "en":
        return {
            "source_text": source_text,
            "detected_language": source_code,
            "translation_needed": False,
            "translated_value_text": source_text,
            "translation_method": "not_required",
            "translation_status": "already_english",
            "translation_error": "",
        }

    payload = {
        "model": OLLAMA_MODEL,
        "messages": [{
            "role": "user",
            "content": build_translation_prompt(
                text=source_text,
                source_code=source_code,
                source_language=source_language,
            ),
        }],
        "stream": False,
        "options": {"temperature": 0},
    }

    async with semaphore:
        for attempt in range(MAX_RETRIES):
            try:
                response = await client.post(
                    f"{OLLAMA_BASE_URL}/api/chat",
                    json=payload,
                )
                response.raise_for_status()

                translated_text = response.json()["message"]["content"].strip()

                if not translated_text:
                    raise ValueError("Empty translation returned")

                commentary_prefixes = [
                    "here's a translation",
                    "here is a translation",
                    "translation:",
                    "the translation is",
                ]

                translated_text_lower = translated_text.lower()

                if any(
                    translated_text_lower.startswith(prefix)
                    for prefix in commentary_prefixes
                ):
                    raise ValueError("Translation returned commentary")

                return {
                    "source_text": source_text,
                    "detected_language": source_code,
                    "translation_needed": True,
                    "translated_value_text": translated_text,
                    "translation_method": f"ollama:{OLLAMA_MODEL}",
                    "translation_status": "translated",
                    "translation_error": "",
                }

            except Exception as error:
                if attempt == MAX_RETRIES - 1:
                    return {
                        "source_text": source_text,
                        "detected_language": source_code,
                        "translation_needed": True,
                        "translated_value_text": None,
                        "translation_method": f"ollama:{OLLAMA_MODEL}",
                        "translation_status": "failed",
                        "translation_error": type(error).__name__,
                    }

                await asyncio.sleep(2 ** attempt)


async def run_translation(texts: list[str]) -> list[dict[str, Any]]:
    semaphore = asyncio.Semaphore(MAX_CONCURRENT_TRANSLATIONS)

    async with httpx.AsyncClient(
        timeout=TRANSLATION_TIMEOUT_SECONDS,
    ) as client:
        health_response = await client.get(f"{OLLAMA_BASE_URL}/api/tags")
        health_response.raise_for_status()

        tasks = [
            translate_value(
                text=text,
                client=client,
                semaphore=semaphore,
            )
            for text in texts
        ]

        return await asyncio.gather(*tasks)


## 14. Run translation and export


In [ ]:
df_values_translated = production_final_df.loc[
    production_final_df["candidate_error"].fillna("").eq("")
    & production_final_df["candidate_text"].fillna("").str.strip().ne("")
    & production_final_df["quality_status"].eq("accepted")
].copy()

df_values_translated["_translation_key"] = (
    df_values_translated["candidate_text"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

unique_texts = (
    df_values_translated["_translation_key"]
    .drop_duplicates()
    .tolist()
)

print("Rows selected:", len(df_values_translated))
print("Unique texts:", len(unique_texts))
if TRANSLATION_CACHE_CSV.exists() and not RERUN_TRANSLATION:
    translation_cache_df = pd.read_csv(TRANSLATION_CACHE_CSV)
else:
    translation_cache_df = pd.DataFrame(
        columns=[
            "source_text",
            "detected_language",
            "translation_needed",
            "translated_value_text",
            "translation_method",
            "translation_status",
            "translation_error",
        ]
    )

cached_texts = set(
    translation_cache_df["source_text"]
    .dropna()
    .astype(str)
)

texts_to_translate = [
    text
    for text in unique_texts
    if text not in cached_texts
]

print(
    "Cached translations:",
    len(unique_texts) - len(texts_to_translate),
)
print("New translations required:", len(texts_to_translate))

if texts_to_translate:
    new_translation_results = await run_translation(
        texts_to_translate
    )
    new_translation_results_df = pd.DataFrame(
        new_translation_results
    )

    translation_cache_df = pd.concat(
        [
            translation_cache_df,
            new_translation_results_df,
        ],
        ignore_index=True,
    ).drop_duplicates(
        subset=["source_text"],
        keep="last",
    )

    translation_cache_df.to_csv(
        TRANSLATION_CACHE_CSV,
        index=False,
        encoding="utf-8-sig",
    )

translation_results_df = translation_cache_df[
    translation_cache_df["source_text"].isin(unique_texts)
].copy()

df_values_translated = (
    df_values_translated.merge(
        translation_results_df,
        left_on="_translation_key",
        right_on="source_text",
        how="left",
        validate="many_to_one",
    )
    .drop(columns=["_translation_key", "source_text"])
)
if "size_bucket" not in df_values_translated.columns:
    df_values_translated = df_values_translated.merge(
        df_companies[
            [
                "company_id",
                "size_bucket",
            ]
        ].drop_duplicates(subset=["company_id"]),
        on="company_id",
        how="left",
        validate="many_to_one",
    )
df_export = (
    df_values_translated[
        [
            "company_id",
            "name",
            "country",
            "size_bucket",
            "internal_url",
            "depth",
            "candidate_section_heading",
            "heading_matched_theme",
            "heading_similarity_score",
            "section_positive_score",
            "section_negative_score",
            "section_semantic_margin",
            "candidate_section_count",
            "candidate_text",
            "quality_status",
            "contains_value_signal",
            "detected_language",
            "translation_needed",
            "translated_value_text",
            "translation_method",
            "translation_status",
            "translation_error",
        ]
    ]
    .rename(columns={
        "company_id": "id",
        "detected_language": "detect_language",
    })
    .copy()
)

text_columns = df_export.select_dtypes(include=["object", "string"]).columns

for column in text_columns:
    df_export[column] = (
        df_export[column]
        .astype("string")
        .str.replace(r"[\r\n\t]+", " ", regex=True)
        .str.replace(r"\s{2,}", " ", regex=True)
        .str.strip()
    )

df_export.to_csv(
    TRANSLATION_OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig",
    quoting=csv.QUOTE_MINIMAL,
    lineterminator="\n",
)

print(f"Saved {len(df_export)} rows to {TRANSLATION_OUTPUT_CSV}")


Rows selected: 2
Unique texts: 2
Cached translations: 2
New translations required: 0
Saved 2 rows to ../data/processed/production_500_sample.csv


## TIME TAKEN TRIAL_1: 51m 44.4s


## TIME TAKEN TRIAL_2: 50m 44.1s


## TIME TAKE TRIAL 3:


In [ ]:
# Final internal links actually selected for translation
preview_df = production_final_df[
    [
        "company_id",
        "name",
        "country",
        "internal_url",
        "depth",
        "discovery_score",
        "matched_theme",
        "content_positive_score",
        "semantic_margin",
    ]
].sort_values(
    ["company_id", "depth", "discovery_score"],
    ascending=[True, True, False],
).reset_index(drop=True)

preview_df.head(50)
print("Companies selected:", len(df_companies))
print("Companies selected for crawl:", len(df_crawl_input))
print("Companies crawled:", df_crawled_pages["company_id"].nunique())

print(
    "Pages with no crawl error:",
    df_crawled_pages["crawl_error"].fillna("").eq("").sum(),
)

print(
    "Pages passing semantic thresholds:",
    len(
        df_crawled_pages.loc[
            df_crawled_pages["crawl_error"].fillna("").eq("")
            & df_crawled_pages["page_text"].fillna("").str.strip().ne("")
            & (df_crawled_pages["content_positive_score"] >= MIN_CONTENT_SIMILARITY)
            & (df_crawled_pages["semantic_margin"] >= MIN_SEMANTIC_MARGIN)
        ]
    ),
)

print("Final accepted companies:", len(production_final_df))


Companies selected: 20
Companies selected for crawl: 19
Companies crawled: 19
Pages with no crawl error: 145
Pages passing semantic thresholds: 10
Final accepted companies: 2


## 15. Inspect targeted multilingual sections


In [ ]:
display(
    df_crawled_pages.loc[
        df_crawled_pages["candidate_section_text"]
        .fillna("")
        .str.strip()
        .ne(""),
        [
            "company_id",
            "country",
            "url",
            "candidate_section_heading",
            "heading_matched_theme",
            "heading_similarity_score",
            "section_positive_score",
            "section_negative_score",
            "section_semantic_margin",
            "candidate_section_text",
        ],
    ]
    .sort_values(
        "section_semantic_margin",
        ascending=False,
    )
    .head(100)
)


,company_id,country,url,candidate_section_heading,heading_matched_theme,heading_similarity_score,section_positive_score,section_negative_score,section_semantic_margin,candidate_section_text
106,C0989,saudi arabia,https://www.daa-international.com/about/what-w...,Our Values,our values,0.992961,0.493947,0.287870,0.206077,Our Values A global airport operator that deli...
62,C0615,germany,http://www.lukas-gesellschaft.de,Weil wir Menschen lieben!,our identity,0.528905,0.361752,0.170728,0.191023,Weil wir Menschen lieben! Ein Blick hinter die...
65,C0615,germany,https://www.lukas-gesellschaft.de,Weil wir Menschen lieben!,our identity,0.528905,0.361752,0.170728,0.191023,Weil wir Menschen lieben! Ein Blick hinter die...
59,C0615,germany,https://www.lukas-gesellschaft.de/startseite-s...,Weil wir Menschen lieben!,our identity,0.528905,0.361752,0.170728,0.191023,Weil wir Menschen lieben! Ein Blick hinter die...
105,C0989,saudi arabia,https://www.daa-international.com,Our customer is at the forefront of everything...,company culture,0.469274,0.495751,0.328580,0.167170,Our customer is at the forefront of everything...
...,...,...,...,...,...,...,...,...,...,...
97,C0839,japan,https://rion-sv.com/index.html,Product information Application Search,company purpose,0.423299,0.188275,0.367233,-0.178957,Product information Application Search Environ...
95,C0839,japan,https://rion-sv.com,Product information Application Search,company purpose,0.423299,0.188275,0.367233,-0.178957,Product information Application Search Environ...
46,C0539,france,https://dromy.fr/mentions-legales,Mentions légales,our principles,0.510063,0.363335,0.591883,-0.228548,Mentions légales CONDITIONS GÉNÉRALES D'UTILIS...
56,C0603,germany,https://www.sprint.de/de/lieferkettensorgfalts...,Allgemeine Informationspflicht gemäß § 36 VSBG,our principles,0.460731,0.281885,0.593465,-0.311580,Allgemeine Informationspflicht gemäß § 36 VSBG...


In [ ]:
print("Companies selected:", len(df_companies))
print("Companies crawled:", df_crawled_pages["company_id"].nunique())

print(
    "Pages with extracted sections:",
    df_crawled_pages[
        "candidate_section_text"
    ].fillna("").str.strip().ne("").sum(),
)

print(
    "Companies with extracted sections:",
    df_crawled_pages.loc[
        df_crawled_pages[
            "candidate_section_text"
        ].fillna("").str.strip().ne(""),
        "company_id",
    ].nunique(),
)

print("Final accepted companies:", len(production_final_df))

Companies selected: 20
Companies crawled: 19
Pages with extracted sections: 70
Companies with extracted sections: 15
Final accepted companies: 2


In [ ]:
review_sections_df = df_crawled_pages.loc[
    df_crawled_pages[
        "candidate_section_text"
    ].fillna("").str.strip().ne("")
].copy()

review_sections_df["passes_rule_1"] = (
    (review_sections_df["heading_similarity_score"] >= 0.50)
    & (review_sections_df["section_positive_score"] >= 0.44)
    & (review_sections_df["section_semantic_margin"] >= 0.08)
)

review_sections_df["passes_rule_2"] = (
    (review_sections_df["heading_similarity_score"] >= 0.42)
    & (review_sections_df["section_positive_score"] >= 0.48)
    & (review_sections_df["section_semantic_margin"] >= 0.12)
)

review_sections_df["contains_value_signal"] = (
    (review_sections_df["section_positive_score"] >= 0.46)
    & (review_sections_df["section_semantic_margin"] >= 0.10)
)

review_sections_df["review_score"] = (
    review_sections_df["section_semantic_margin"]
    + (0.15 * review_sections_df["heading_similarity_score"])
    + (0.10 * review_sections_df["section_positive_score"])
)

best_section_per_company_df = (
    review_sections_df
    .sort_values(
        [
            "company_id",
            "contains_value_signal",
            "review_score",
        ],
        ascending=[True, False, False],
    )
    .drop_duplicates(
        subset=["company_id"],
        keep="first",
    )
    .reset_index(drop=True)
)

best_rejected_sections_df = (
    best_section_per_company_df.loc[
        ~best_section_per_company_df["contains_value_signal"]
    ]
    .copy()
)

display(
    best_rejected_sections_df[
        [
            "company_id",
            "name",
            "country",
            "url",
            "candidate_section_heading",
            "heading_matched_theme",
            "heading_similarity_score",
            "section_positive_score",
            "section_negative_score",
            "section_semantic_margin",
            "candidate_section_text",
        ]
    ]
)

,company_id,name,country,url,candidate_section_heading,heading_matched_theme,heading_similarity_score,section_positive_score,section_negative_score,section_semantic_margin,candidate_section_text
0,C0050,southern steel group,australia,https://www.southernsteelgroup.com.au/company-...,KomoMetform Engineering,guiding principles,0.418962,0.305973,0.226999,0.078975,KomoMetform Engineering Location: Victoria Spe...
1,C0142,the rabbit sanctuary,australia,https://www.rabbitsanctuary.com.au/community,"""I alone cannot change the world, but I can ca...",our philosophy,0.444768,0.181022,0.212453,-0.031431,"""I alone cannot change the world, but I can ca..."
3,C0248,aliança sul,brazil,https://aliancasul.com.br/trabalhe-conosco,Conheça oportunidades no Empresarial,company purpose,0.600383,0.481639,0.469407,0.012232,Conheça oportunidades no Empresarial Buscando ...
4,C0539,dromy 💚 le dernier kilomètre responsable,france,https://dromy.fr,Une vision durable,our vision,0.760615,0.288233,0.278950,0.009283,Une vision durable Faire en sorte que tout le ...
5,C0603,sprint sanierung gmbh,germany,https://www.sprint.de/de/geschaeftskunden/wie-...,Qualität – Gutes lohnt sich,company values,0.492189,0.432465,0.305661,0.126803,Qualität – Gutes lohnt sich Ist Qualität ein C...
6,C0615,st. josefs-hospital dortmund,germany,https://www.lukas-gesellschaft.de/startseite-s...,Weil wir Menschen lieben!,our identity,0.528905,0.361752,0.170728,0.191023,Weil wir Menschen lieben! Ein Blick hinter die...
7,C0794,hamamatsu corporation,japan,https://www.hamamatsu.com/jp/en/our-company/ou...,The spirit of pursuing the unknown and unexplo...,our ethos,0.555680,0.297310,0.210711,0.086599,The spirit of pursuing the unknown and unexplo...
8,C0839,"rion co., ltd environmental instruments",japan,https://rion-sv.com/support/index.html,YOUTUBE,our vision,0.408269,0.378260,0.515071,-0.136811,YOUTUBE The features/functions/how to measure/...
9,C0875,vogaro株式会社,japan,https://www.vogaro.co.jp/about,大阪本社オフィス,company mission,0.408214,0.160772,0.315259,-0.154487,大阪本社オフィス 〒530-8304 大阪府大阪市北区茶屋町 17-1 （MBS メディアホ...
11,C1051,litter4tokens,south africa,https://www.litter4tokens.org/community,Token Shops,company culture,0.418172,0.146802,0.223865,-0.077064,Token Shops Each token shop relies on food and...
